In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# NB_Silver_Products
# Bronze -> Silver
# ============================================================

SOURCE_TABLE = "bronze_products"
SILVER_TABLE = "silver_products"
QUARANTINE_TABLE = "quarantine_products"

# ------------------------------------------------------------
# 1. Read Bronze
# ------------------------------------------------------------

df = spark.table(SOURCE_TABLE)

# ------------------------------------------------------------
# 2. Standardize types
# ------------------------------------------------------------

df = (
    df
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("product_name", F.trim(F.col("product_name").cast("string")))
    .withColumn("category", F.trim(F.col("category").cast("string")))
    .withColumn("price", F.col("price").cast("decimal(10,2)"))
    .withColumn("stock_quantity", F.col("stock_quantity").cast("int"))
)

# ------------------------------------------------------------
# 3. Validation
# ------------------------------------------------------------

df_checked = (
    df
    .withColumn(
        "dq_reason",
        F.when(
            F.col("product_id").isNull(),
            F.lit("NULL_PRODUCT_ID")
        )
        .when(
            F.col("product_name").isNull() |
            (F.col("product_name") == ""),
            F.lit("NULL_OR_EMPTY_PRODUCT_NAME")
        )
        .when(
            F.col("price").isNull(),
            F.lit("NULL_PRICE")
        )
        .when(
            F.col("price") < 0,
            F.lit("NEGATIVE_PRICE")
        )
        .when(
            F.col("stock_quantity").isNull(),
            F.lit("NULL_STOCK_QUANTITY")
        )
        .when(
            F.col("stock_quantity") < 0,
            F.lit("NEGATIVE_STOCK_QUANTITY")
        )
        .otherwise(F.lit(None))
    )
)

# ------------------------------------------------------------
# 4. Quarantine
# ------------------------------------------------------------

df_quarantine = (
    df_checked
    .filter(F.col("dq_reason").isNotNull())
    .withColumn("quarantined_at", F.current_timestamp())
)

df_quarantine.write.mode("overwrite").format("delta").saveAsTable(
    QUARANTINE_TABLE
)

# ------------------------------------------------------------
# 5. Valid records
# ------------------------------------------------------------

df_valid = (
    df_checked
    .filter(F.col("dq_reason").isNull())
    .drop("dq_reason")
)

# ------------------------------------------------------------
# 6. Deduplicate
# ------------------------------------------------------------

window = Window.partitionBy("product_id").orderBy(
    F.col("product_id")
)

df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

# ------------------------------------------------------------
# 7. Write Silver
# ------------------------------------------------------------

df_dedup.write.mode("overwrite").format("delta").saveAsTable(
    SILVER_TABLE
)

print(f"Silver table created: {SILVER_TABLE}")
print(f"Valid records: {df_dedup.count()}")
print(f"Quarantined records: {df_quarantine.count()}")

StatementMeta(, 65ab2cbe-d96f-4b2c-927f-a6b237eb7f6a, 3, Finished, Available, Finished, False)

Silver table created: silver_products
Valid records: 997
Quarantined records: 2
